# 08. Attention core — DeepSeek-V4-Flash attention schedule

Tensor widths, batch size, and test sequence length are reduced. The notebook implements 43 attention sites, 64 query heads, one shared KV head, SWA-128, CSA-4, HCA-128, a 64-head top-512 Lightning Indexer, 8-way grouped output projection, partial RoPE, and separate main/compressed-branch rotary frequencies.


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cpu")
torch.set_num_threads(min(2, torch.get_num_threads()))


In [ ]:
V4_FLASH_ALL_COMPRESS_RATIOS = [0, 0]
for _ in range(20):
    V4_FLASH_ALL_COMPRESS_RATIOS.extend([4, 128])
V4_FLASH_ALL_COMPRESS_RATIOS.extend([4, 0])

V4_FLASH_COMPRESS_RATIOS = V4_FLASH_ALL_COMPRESS_RATIOS[:-1]
V4_FLASH_MTP_COMPRESS_RATIO = V4_FLASH_ALL_COMPRESS_RATIOS[-1]

V4_QUERY_HEADS = 64
V4_KV_HEADS = 1
V4_SLIDING_WINDOW = 128
V4_INDEX_HEADS = 64
V4_INDEX_TOPK = 512
V4_OUTPUT_GROUPS = 8

MAIN_ROPE_THETA = 10_000.0
COMPRESS_ROPE_THETA = 160_000.0
YARN_FACTOR = 16.0
YARN_BETA_FAST = 32.0
YARN_BETA_SLOW = 1.0
YARN_ORIGINAL_MAX_POSITION = 65_536

assert len(V4_FLASH_ALL_COMPRESS_RATIOS) == 44
assert len(V4_FLASH_COMPRESS_RATIOS) == 43
assert V4_FLASH_COMPRESS_RATIOS[:4] == [0, 0, 4, 128]
assert V4_FLASH_MTP_COMPRESS_RATIO == 0


## 1. Main RoPE and compressed-branch YaRN RoPE

Pure sliding-window sites use ordinary RoPE with theta=10000. CSA/HCA compressed entries use theta=160000 with YaRN factor 16.


In [ ]:
def _find_correction_dim(num_rotations, dim, base, original_max_position):
    return (
        dim
        * math.log(original_max_position / (num_rotations * 2 * math.pi))
        / (2 * math.log(base))
    )


def _linear_ramp(low, high, count, device):
    if low == high:
        high += 0.001
    index = torch.arange(count, device=device, dtype=torch.float32)
    ramp = (index - low) / (high - low)
    return ramp.clamp(0.0, 1.0)


def yarn_inverse_frequency(rope_dim, device, base=COMPRESS_ROPE_THETA, factor=YARN_FACTOR, beta_fast=YARN_BETA_FAST, beta_slow=YARN_BETA_SLOW, original_max_position=YARN_ORIGINAL_MAX_POSITION):
    pair_index = torch.arange(0, rope_dim, 2, device=device, dtype=torch.float32)
    position_frequencies = base ** (pair_index / rope_dim)
    inv_extrapolation = 1.0 / position_frequencies
    inv_interpolation = 1.0 / (factor * position_frequencies)
    low = math.floor(_find_correction_dim(beta_fast, rope_dim, base, original_max_position))
    high = math.ceil(_find_correction_dim(beta_slow, rope_dim, base, original_max_position))
    low = max(low, 0)
    high = min(high, rope_dim - 1)
    extrapolation_factor = 1.0 - _linear_ramp(low, high, rope_dim // 2, device)
    return inv_interpolation * (1.0 - extrapolation_factor) + inv_extrapolation * extrapolation_factor


def standard_inverse_frequency(rope_dim, device, base=MAIN_ROPE_THETA):
    pair_index = torch.arange(0, rope_dim, 2, device=device, dtype=torch.float32)
    return 1.0 / (base ** (pair_index / rope_dim))


def rotate_rope_subspace(x, position_ids, rope_dim, *, compressed=False, conjugate=False):
    if rope_dim == 0:
        return x
    content = x[..., :-rope_dim]
    rotary = x[..., -rope_dim:]
    inv_freq = yarn_inverse_frequency(rope_dim, x.device) if compressed else standard_inverse_frequency(rope_dim, x.device)
    angles = position_ids.float()[:, None] * inv_freq[None]
    cosine = angles.cos()
    sine = angles.sin()
    if rotary.ndim == 4:
        cosine = cosine[None, :, None]
        sine = sine[None, :, None]
    elif rotary.ndim == 3:
        cosine = cosine[None]
        sine = sine[None]
    else:
        raise ValueError(f"unsupported RoPE rank: {rotary.ndim}")
    if conjugate:
        sine = -sine
    even = rotary[..., 0::2]
    odd = rotary[..., 1::2]
    rotated = torch.stack([even * cosine - odd * sine, even * sine + odd * cosine], dim=-1).flatten(-2)
    return torch.cat([content, rotated], dim=-1)

rope_dim = 8
main_frequency = standard_inverse_frequency(rope_dim, device)
compress_frequency = yarn_inverse_frequency(rope_dim, device)
assert not torch.allclose(main_frequency, compress_frequency)


## 2. CSA / HCA compressors and Lightning Indexer


In [ ]:
class Compressor(nn.Module):
    def __init__(self, model_dim, head_dim, rate):
        super().__init__()
        assert rate in {4, 128}
        self.rate = rate
        self.head_dim = head_dim
        projected_dim = 2 * head_dim if rate == 4 else head_dim
        self.kv_projection = nn.Linear(model_dim, projected_dim, bias=False)
        self.gate_projection = nn.Linear(model_dim, projected_dim, bias=False)
        self.position_bias = nn.Parameter(torch.zeros(rate, projected_dim))
        self.norm = nn.RMSNorm(head_dim)

    def forward(self, hidden):
        batch, length, _ = hidden.shape
        usable = (length // self.rate) * self.rate
        hidden = hidden[:, :usable]
        if usable == 0:
            empty = hidden.new_zeros(batch, 0, self.head_dim)
            positions = torch.empty(0, dtype=torch.long, device=hidden.device)
            return empty, positions
        blocks = usable // self.rate
        if self.rate == 128:
            kv = self.kv_projection(hidden).view(batch, blocks, self.rate, self.head_dim)
            gate = self.gate_projection(hidden).view_as(kv) + self.position_bias
            weight = gate.softmax(dim=2, dtype=torch.float32).to(kv.dtype)
            compressed = (kv * weight).sum(dim=2)
        else:
            kv = self.kv_projection(hidden).view(batch, blocks, self.rate, 2 * self.head_dim)
            gate = self.gate_projection(hidden).view_as(kv) + self.position_bias
            ca, cb = kv.chunk(2, dim=-1)
            ga, gb = gate.chunk(2, dim=-1)
            entries = hidden.new_zeros(batch, blocks, 2 * self.rate, self.head_dim)
            logits = hidden.new_full(entries.shape, float("-inf"))
            entries[:, :, self.rate:] = cb
            logits[:, :, self.rate:] = gb
            if blocks > 1:
                entries[:, 1:, :self.rate] = ca[:, :-1]
                logits[:, 1:, :self.rate] = ga[:, :-1]
            weight = logits.softmax(dim=2, dtype=torch.float32).to(entries.dtype)
            compressed = (entries * weight).sum(dim=2)
        positions = torch.arange(blocks, device=hidden.device) * self.rate
        return self.norm(compressed), positions


class LightningIndexer(nn.Module):
    def __init__(self, model_dim=32, q_rank=8, index_heads=64, index_head_dim=2, top_k=512):
        super().__init__()
        assert index_heads == 64
        assert top_k == 512
        self.index_heads = index_heads
        self.index_head_dim = index_head_dim
        self.top_k = top_k
        self.compressor = Compressor(model_dim, index_head_dim, rate=4)
        self.query_projection = nn.Linear(q_rank, index_heads * index_head_dim, bias=False)
        self.head_weight = nn.Linear(model_dim, index_heads, bias=False)

    def forward(self, hidden, q_latent):
        batch, length, _ = hidden.shape
        token_positions = torch.arange(length, device=hidden.device)
        keys, key_positions = self.compressor(hidden)
        if keys.size(1) == 0:
            return torch.empty(batch, length, 0, dtype=torch.long, device=hidden.device)
        keys = rotate_rope_subspace(keys[:, :, None], key_positions, self.index_head_dim, compressed=True).squeeze(2)
        queries = self.query_projection(q_latent).view(batch, length, self.index_heads, self.index_head_dim)
        queries = rotate_rope_subspace(queries, token_positions, self.index_head_dim, compressed=True)
        per_head = torch.einsum("bthd,bnd->bthn", queries.float(), keys.float())
        per_head = F.relu(per_head) / math.sqrt(self.index_head_dim)
        head_weight = self.head_weight(hidden).float() / math.sqrt(self.index_heads)
        score = (per_head * head_weight[..., None]).sum(dim=2)
        causal_count = (token_positions[None] + 1) // 4
        entry_ids = torch.arange(keys.size(1), device=hidden.device)
        valid = entry_ids[None, None] < causal_count[..., None]
        score = score.masked_fill(~valid, float("-inf"))
        runtime_top_k = min(self.top_k, keys.size(1))
        selected = score.topk(runtime_top_k, dim=-1).indices
        selected_valid = selected < causal_count[..., None]
        return torch.where(selected_valid, selected, -torch.ones_like(selected))


## 3. 43-site attention schedule


In [ ]:
class GroupedOutputProjection(nn.Module):
    def __init__(self, query_heads=64, head_dim=4, groups=8, low_rank=16, model_dim=32):
        super().__init__()
        total = query_heads * head_dim
        assert total % groups == 0
        self.groups = groups
        group_width = total // groups
        self.down = nn.ModuleList([nn.Linear(group_width, low_rank, bias=False) for _ in range(groups)])
        self.up = nn.Linear(groups * low_rank, model_dim, bias=False)
    def forward(self, heads):
        flattened = heads.flatten(2)
        chunks = flattened.chunk(self.groups, dim=-1)
        latent = [projection(chunk) for projection, chunk in zip(self.down, chunks)]
        return self.up(torch.cat(latent, dim=-1))


class DeepSeekV4Attention(nn.Module):
    def __init__(self, compression_ratio, model_dim=32, query_heads=64, head_dim=4, rope_dim=2, q_rank=8):
        super().__init__()
        assert compression_ratio in {0, 4, 128}
        assert query_heads == 64
        self.compression_ratio = compression_ratio
        self.query_heads = query_heads
        self.head_dim = head_dim
        self.rope_dim = rope_dim
        self.sliding_window = 128
        self.q_down = nn.Linear(model_dim, q_rank, bias=False)
        self.q_norm = nn.RMSNorm(q_rank)
        self.q_up = nn.Linear(q_rank, query_heads * head_dim, bias=False)
        self.shared_kv = nn.Linear(model_dim, head_dim, bias=False)
        self.shared_kv_norm = nn.RMSNorm(head_dim)
        self.attention_sink = nn.Parameter(torch.zeros(query_heads))
        self.output_projection = GroupedOutputProjection(query_heads=query_heads, head_dim=head_dim, groups=8, low_rank=16, model_dim=model_dim)
        self.compressor = Compressor(model_dim, head_dim, rate=compression_ratio) if compression_ratio else None
        self.indexer = LightningIndexer(model_dim=model_dim, q_rank=q_rank, index_heads=64, index_head_dim=2, top_k=512) if compression_ratio == 4 else None

    def forward(self, hidden):
        batch, length, _ = hidden.shape
        positions = torch.arange(length, device=hidden.device)
        q_latent = self.q_norm(self.q_down(hidden))
        query = self.q_up(q_latent).view(batch, length, self.query_heads, self.head_dim)
        query = F.rms_norm(query, (self.head_dim,))
        query = rotate_rope_subspace(query, positions, self.rope_dim, compressed=False)
        local_kv = self.shared_kv_norm(self.shared_kv(hidden))
        local_kv = rotate_rope_subspace(local_kv[:, :, None], positions, self.rope_dim, compressed=False).squeeze(2)
        compressed = None
        selected = None
        if self.compressor is not None:
            compressed, compressed_positions = self.compressor(hidden)
            compressed = rotate_rope_subspace(compressed[:, :, None], compressed_positions, self.rope_dim, compressed=True).squeeze(2)
        if self.indexer is not None:
            selected = self.indexer(hidden, q_latent)
        outputs = []
        batch_ids = torch.arange(batch, device=hidden.device)[:, None]
        for token_index in range(length):
            local_start = max(0, token_index - self.sliding_window + 1)
            kv_parts = [local_kv[:, local_start:token_index + 1]]
            valid_parts = [torch.ones(batch, token_index - local_start + 1, dtype=torch.bool, device=hidden.device)]
            if self.compression_ratio == 4 and selected is not None and selected.size(-1) > 0:
                ids = selected[:, token_index]
                valid = ids >= 0
                safe = ids.clamp_min(0)
                kv_parts.append(compressed[batch_ids, safe])
                valid_parts.append(valid)
            if self.compression_ratio == 128 and compressed is not None and compressed.size(1) > 0:
                causal_count = (token_index + 1) // 128
                kv_parts.append(compressed)
                ids = torch.arange(compressed.size(1), device=hidden.device)
                valid_parts.append((ids[None] < causal_count).expand(batch, -1))
            kv = torch.cat(kv_parts, dim=1)
            valid = torch.cat(valid_parts, dim=1)
            score = torch.einsum("bhd,bkd->bhk", query[:, token_index], kv) / math.sqrt(self.head_dim)
            score = score.masked_fill(~valid[:, None], torch.finfo(score.dtype).min)
            sink = self.attention_sink[None, :, None].expand(batch, -1, 1)
            logits = torch.cat([score, sink], dim=-1)
            weight = logits.softmax(dim=-1)[..., :-1]
            outputs.append(torch.einsum("bhk,bkd->bhd", weight, kv))
        heads = torch.stack(outputs, dim=1)
        heads = rotate_rope_subspace(heads, positions, self.rope_dim, compressed=False, conjugate=True)
        return self.output_projection(heads)


class DeepSeekV4FlashAttentionStack(nn.Module):
    def __init__(self, model_dim=32):
        super().__init__()
        self.compress_ratios = list(V4_FLASH_COMPRESS_RATIOS)
        self.layers = nn.ModuleList([DeepSeekV4Attention(compression_ratio=ratio, model_dim=model_dim) for ratio in self.compress_ratios])
    def forward(self, hidden):
        for layer in self.layers:
            hidden = hidden + layer(hidden)
        return hidden

model = DeepSeekV4FlashAttentionStack().to(device)
assert len(model.layers) == 43
assert model.compress_ratios == V4_FLASH_COMPRESS_RATIOS
assert all(layer.query_heads == 64 for layer in model.layers)
assert all(layer.sliding_window == 128 for layer in model.layers)
assert all(layer.output_projection.groups == 8 for layer in model.layers)
csa_layers = [layer for layer in model.layers if layer.compression_ratio == 4]
hca_layers = [layer for layer in model.layers if layer.compression_ratio == 128]
assert all(layer.indexer.top_k == 512 for layer in csa_layers)
assert all(layer.compressor.rate == 4 for layer in csa_layers)
assert all(layer.compressor.rate == 128 for layer in hca_layers)
hidden = torch.randn(1, 4, 32, device=device)
output = model(hidden)
output.square().mean().backward()
assert output.shape == hidden.shape
